In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['OMP_NUM_THREADS'] = '1'
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('../').resolve()
sys.path.append(str(PROJECT_ROOT))

from src.utils.utils import load_csv
from src.models.train_bert import BERTTrainer

from src.config import (
    CLEANED_TRAIN_PATH,
    CLEANED_TEST_PATH,
    TRAIN_LABEL_PATH,
    SUBMISSION_BERT_PATH,
)

Using device: cpu


/Users/nhatnam/Documents/DM_252/Assignment/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load data
train_df = load_csv(CLEANED_TRAIN_PATH)
test_df  = load_csv(CLEANED_TEST_PATH)

# Gắn label vào train_df
labels_df = load_csv(TRAIN_LABEL_PATH)
train_df['label'] = labels_df.values.ravel()

print(f'Train shape : {train_df.shape}')
print(f'Test shape  : {test_df.shape}')
print(f'Columns     : {train_df.columns.tolist()}')
print(f'\nLabel distribution (1-5):')
print(train_df['label'].value_counts().sort_index())

Train shape : (2496, 9)
Test shape  : (596, 7)
Columns     : ['id', 'title', 'venue', 'year', 'authors', 'doi', 'Label', 'abstract', 'label']

Label distribution (1-5):
label
1    904
2    514
3    439
4    367
5    272
Name: count, dtype: int64


In [ ]:
print('\n====================')
print('MODEL: SciBERT + Ordinal Classification')
print('====================')

# Khởi tạo Trainer
trainer = BERTTrainer(
)


MODEL: SciBERT + Ordinal Classification


In [4]:
# Huấn luyện 5-fold Stratified CV
macro_f1_cv = trainer.train(train_df, label_col='label')

print(f'CV Macro F1-Score: {macro_f1_cv:.4f}')


Fold 1/5
  Epoch  1/17 loss=2.3414 val_macro_f1=0.1063 ✔ best
  Epoch  2/17 loss=2.1101 val_macro_f1=0.2994 ✔ best
  Epoch  3/17 loss=1.7655 val_macro_f1=0.3702 ✔ best
  Epoch  4/17 loss=1.5883 val_macro_f1=0.3740 ✔ best
  Epoch  5/17 loss=1.4079 val_macro_f1=0.4081 ✔ best
  Epoch  6/17 loss=1.2240 val_macro_f1=0.3972 (no improve 1/5)
  Epoch  7/17 loss=1.0652 val_macro_f1=0.4046 (no improve 2/5)
  Epoch  8/17 loss=0.9099 val_macro_f1=0.4130 ✔ best
  Epoch  9/17 loss=0.7792 val_macro_f1=0.4061 (no improve 1/5)
  Epoch 10/17 loss=0.6673 val_macro_f1=0.4156 ✔ best
  Epoch 11/17 loss=0.6102 val_macro_f1=0.4168 ✔ best
  Epoch 12/17 loss=0.5837 val_macro_f1=0.4139 (no improve 1/5)
  Epoch 13/17 loss=0.5207 val_macro_f1=0.4112 (no improve 2/5)
  Epoch 14/17 loss=0.5014 val_macro_f1=0.4203 ✔ best
  Epoch 15/17 loss=0.4751 val_macro_f1=0.4193 (no improve 1/5)
  Epoch 16/17 loss=0.4710 val_macro_f1=0.4185 (no improve 2/5)
  Epoch 17/17 loss=0.4532 val_macro_f1=0.4220 ✔ best
  → Best val Macro 

In [5]:
# Lưu checkpoints để dùng lại, không cần train lại
trainer.save_models(str(PROJECT_ROOT / 'models' / 'saved' / 'bert_folds'))
print('✔ Saved all fold checkpoints')

Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_1.pt
Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_2.pt
Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_3.pt
Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_4.pt
Saved: /Users/nhatnam/Documents/DM_252/Assignment/models/saved/bert_folds/bert_fold_5.pt
✔ Saved all fold checkpoints


In [6]:
# Dự đoán trên tập Test
# Ensemble softmax avg từ tất cả fold models → nhãn 1-5
y_test_pred = trainer.predict(test_df)

print(f'Predictions shape: {y_test_pred.shape}')
print(f'Predicted classes : {np.unique(y_test_pred)}')

Predictions shape: (596,)
Predicted classes : [1 2 3 4 5]


In [7]:
# Tạo và lưu submission
submission = pd.DataFrame({
    'id'   : test_df['id'],
    'Label': y_test_pred,
})

os.makedirs(Path(SUBMISSION_BERT_PATH).parent, exist_ok=True)
submission.to_csv(SUBMISSION_BERT_PATH, index=False)
print(f'✔ Saved BERT submission → {SUBMISSION_BERT_PATH}')

print('\nPredicted labels distribution (Should be 1-5):')
print(submission['Label'].value_counts().sort_index())

✔ Saved BERT submission → /Users/nhatnam/Documents/DM_252/Assignment/data/submission/submission_bert.csv

Predicted labels distribution (Should be 1-5):
Label
1    186
2    184
3    120
4     77
5     29
Name: count, dtype: int64


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

def diagnose(df, label_col="label", n_splits=5):
    labels = df[label_col].values
    
    # 1. Phân phối tổng thể
    print("=== PHÂN PHỐI LABEL ===")
    dist = pd.Series(labels).value_counts().sort_index()
    for k, v in dist.items():
        bar = "█" * int(v / dist.max() * 30)
        print(f"  Label {k}: {v:4d} ({v/len(labels)*100:.1f}%)  {bar}")
    print(f"  Imbalance ratio (max/min): {dist.max()/dist.min():.1f}x")
    
    # 2. CV fold distribution
    print("\n=== CV FOLD DISTRIBUTION ===")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_dists = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, labels)):
        val_dist = pd.Series(labels[val_idx]).value_counts().sort_index()
        fold_dists.append(val_dist)
        print(f"  Fold {fold+1} val: {val_dist.to_dict()}  (n={len(val_idx)})")
    
    # Check độ lệch giữa các fold
    fold_df = pd.DataFrame(fold_dists).fillna(0)
    cv_std = fold_df.std(axis=0)
    print(f"\n  Std across folds per label: {cv_std.round(1).to_dict()}")
    if cv_std.max() > 5:
        print("  ⚠️  Có fold lệch đáng kể — dataset nhỏ hoặc label hiếm")
    else:
        print("  ✅ Các fold khá đồng đều")
    
    # 3. Adjacency check — label có thực sự ordinal không?
    print("\n=== ORDINAL SANITY CHECK ===")
    print("  (Nếu label thực sự ordinal, các feature liên tục nên tăng/giảm đều theo label)")
    print(f"  Min label: {labels.min()}, Max: {labels.max()}, Unique: {sorted(np.unique(labels))}")

diagnose(train_df)

=== PHÂN PHỐI LABEL ===
  Label 1:  904 (36.2%)  ██████████████████████████████
  Label 2:  514 (20.6%)  █████████████████
  Label 3:  439 (17.6%)  ██████████████
  Label 4:  367 (14.7%)  ████████████
  Label 5:  272 (10.9%)  █████████
  Imbalance ratio (max/min): 3.3x

=== CV FOLD DISTRIBUTION ===
  Fold 1 val: {1: 181, 2: 103, 3: 88, 4: 73, 5: 55}  (n=500)
  Fold 2 val: {1: 181, 2: 103, 3: 88, 4: 73, 5: 54}  (n=499)
  Fold 3 val: {1: 181, 2: 103, 3: 87, 4: 74, 5: 54}  (n=499)
  Fold 4 val: {1: 181, 2: 102, 3: 88, 4: 74, 5: 54}  (n=499)
  Fold 5 val: {1: 180, 2: 103, 3: 88, 4: 73, 5: 55}  (n=499)

  Std across folds per label: {1: 0.4, 2: 0.4, 3: 0.4, 4: 0.5, 5: 0.5}
  ✅ Các fold khá đồng đều

=== ORDINAL SANITY CHECK ===
  (Nếu label thực sự ordinal, các feature liên tục nên tăng/giảm đều theo label)
  Min label: 1, Max: 5, Unique: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
